In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import os

META_PATH = "/Users/adiallo/Desktop/Thesis/Data_Documents/data_all.csv"
SAMPLE_DIR = "/Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples"

meta = pd.read_csv(META_PATH)
meta['deident'] = meta['deident'].astype(str)

def get_mets(row):
    dist = str(row.get('Distant_Mets', 'False')).upper()
    ln = str(row.get('ln_only', 'False')).upper()
    if dist in ['TRUE', 'T']: return 'Distant_Mets'
    elif ln in ['TRUE', 'T']: return 'LN_Mets'
    return 'No_Mets'

meta['mets_group'] = meta.apply(get_mets, axis=1)

results = []

for _, row in meta.iterrows():
    sid = str(row['deident'])
    path = os.path.join(SAMPLE_DIR, f"{sid}.h5ad")
    if not os.path.exists(path):
        continue
    
    adata = sc.read_h5ad(path)
    
    n_spots = adata.n_obs
    n_genes = adata.n_vars
    
    # Cells per spot from Cell2Location
    abund = adata.obsm['means_cell_abundance_w_sf']
    cells_per_spot = abund.sum(axis=1)
    
    # Region counts if available
    if 'region' in adata.obs.columns:
        n_ct = (adata.obs['region'] == 'CT').sum()
        n_im = (adata.obs['region'] == 'IM').sum()
        n_stroma = (adata.obs['region'] == 'Stroma').sum()
    else:
        hist = adata.obs['histology'].values
        n_ct = sum(h == 'cancer_cancer' for h in hist)
        n_im = 0
        n_stroma = sum(h in ['benign_stroma', 'benign_fat', 'inter', 'stroma', 'benign_muscularis'] for h in hist)
    
    # QC metrics if available
    if 'total_counts' in adata.obs.columns:
        median_counts = adata.obs['total_counts'].median()
        median_genes = adata.obs['n_genes_by_counts'].median()
    else:
        # Compute on the fly
        if hasattr(adata.X, 'toarray'):
            total = np.array(adata.X.sum(axis=1)).flatten()
            n_genes_per_spot = np.array((adata.X > 0).sum(axis=1)).flatten()
        else:
            total = adata.X.sum(axis=1)
            n_genes_per_spot = (adata.X > 0).sum(axis=1)
        median_counts = np.median(total)
        median_genes = np.median(n_genes_per_spot)
    
    results.append({
        'Sample': sid,
        'Mets_Group': row['mets_group'],
        'Grade': row.get('Grade', ''),
        'Site': row.get('site', ''),
        'Age': row.get('age', ''),
        'Sex': row.get('sex', ''),
        'MLH1': row.get('MLH1', ''),
        'N_Spots': n_spots,
        'N_Genes': n_genes,
        'Median_Counts_Per_Spot': int(median_counts),
        'Median_Genes_Per_Spot': int(median_genes),
        'Mean_Cells_Per_Spot': round(cells_per_spot.mean(), 1),
        'Median_Cells_Per_Spot': round(cells_per_spot.median(), 1),
        'N_CT': n_ct,
        'N_IM': n_im,
        'N_Stroma': n_stroma,
    })
    
    print(f"  {sid}: {n_spots} spots, {cells_per_spot.mean():.1f} cells/spot")

df = pd.DataFrame(results)
df = df.sort_values('Sample', key=lambda x: x.astype(int))

# Save
out_path = "/Users/adiallo/Desktop/Thesis/Results/AIM_2_Results/sample_overview_table.csv"
df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

# Print summary stats
print(f"\n{'='*50}")
print(f"Total samples: {len(df)}")
print(f"Mets groups: {df['Mets_Group'].value_counts().to_dict()}")
print(f"Spots: {df['N_Spots'].sum():,} total, {df['N_Spots'].median():.0f} median per sample")
print(f"Cells per spot: {df['Mean_Cells_Per_Spot'].mean():.1f} mean across samples")

  0: 6658 spots, 8.4 cells/spot
  3: 5571 spots, 8.8 cells/spot
  6: 6414 spots, 9.0 cells/spot
  10: 6138 spots, 8.5 cells/spot
  11: 7353 spots, 8.3 cells/spot
  13: 5910 spots, 9.0 cells/spot
  14: 6855 spots, 9.0 cells/spot
  18: 6319 spots, 8.5 cells/spot
  28: 6989 spots, 9.0 cells/spot
  29: 6768 spots, 9.8 cells/spot
  30: 6476 spots, 8.5 cells/spot
  34: 6297 spots, 8.2 cells/spot
  36: 6873 spots, 9.8 cells/spot
  38: 6971 spots, 8.9 cells/spot
  40: 7217 spots, 8.8 cells/spot
  46: 6826 spots, 8.3 cells/spot
  49: 6502 spots, 9.7 cells/spot
  51: 6342 spots, 8.5 cells/spot
  52: 6718 spots, 8.9 cells/spot
  60: 6492 spots, 9.0 cells/spot
  72: 6706 spots, 8.7 cells/spot
  73: 6532 spots, 8.8 cells/spot
  74: 7010 spots, 8.8 cells/spot
  80: 6526 spots, 8.2 cells/spot
  81: 7274 spots, 8.1 cells/spot
  83: 7485 spots, 9.4 cells/spot
  86: 7480 spots, 8.5 cells/spot
  91: 6651 spots, 9.0 cells/spot
  96: 6987 spots, 8.4 cells/spot
  97: 6142 spots, 9.0 cells/spot
  100: 7194 s

In [2]:
import pandas as pd
import numpy as np
import scanpy as sc
import os

META_PATH = "/Users/adiallo/Desktop/Thesis/Data_Documents/data_all.csv"
SAMPLE_DIR = "/Users/adiallo/Desktop/Thesis/Data_Documents/All_Data/Visium_samples/Visium_Data/samples"
OUTPUT_DIR = "/Users/adiallo/Desktop/Thesis/Results/AIM_2_Results"

meta = pd.read_csv(META_PATH)
meta['deident'] = meta['deident'].astype(str)

visium_ids = sorted([f.replace('.h5ad', '') for f in os.listdir(SAMPLE_DIR)
                     if f.endswith('.h5ad') and f.replace('.h5ad', '').isdigit()])
meta = meta[meta['deident'].isin(visium_ids)].copy()

def get_any_mets(row):
    dist = str(row.get('Distant_Mets', 'False')).upper()
    ln = str(row.get('ln_only', 'False')).upper()
    if dist in ['TRUE', 'T'] or ln in ['TRUE', 'T']:
        return 'Any_Mets'
    return 'No_Mets'

meta['mets_binary'] = meta.apply(get_any_mets, axis=1)
meta['has_distant'] = meta['Distant_Mets'].apply(lambda x: str(x).upper() in ['TRUE', 'T'])
meta['has_ln'] = meta['ln_only'].apply(lambda x: str(x).upper() in ['TRUE', 'T'])

# ================================================================
# Load spatial stats per sample
# ================================================================
print("Loading spatial data per sample...")

sample_stats = []
for sid in visium_ids:
    path = os.path.join(SAMPLE_DIR, f"{sid}.h5ad")
    adata = sc.read_h5ad(path)

    abund = adata.obsm['means_cell_abundance_w_sf']
    cells_per_spot = abund.sum(axis=1)

    # Regions
    if 'region' in adata.obs.columns:
        n_ct = (adata.obs['region'] == 'CT').sum()
        n_im = (adata.obs['region'] == 'IM').sum()
        n_stroma = (adata.obs['region'] == 'Stroma').sum()
    else:
        n_ct = 0; n_im = 0; n_stroma = 0

    # Refined regions
    if 'refined_region' in adata.obs.columns:
        n_tumor_epi = (adata.obs['refined_region'] == 'Tumor_epithelial').sum()
        n_intra_stroma = (adata.obs['refined_region'] == 'Intratumoral_stroma').sum()
        n_normal_stroma = (adata.obs['refined_region'] == 'Normal_stroma').sum()
    else:
        n_tumor_epi = 0; n_intra_stroma = 0; n_normal_stroma = 0

    sample_stats.append({
        'deident': sid,
        'n_spots': adata.n_obs,
        'n_genes': adata.n_vars,
        'mean_cells_per_spot': cells_per_spot.mean(),
        'n_ct': n_ct,
        'n_im': n_im,
        'n_stroma': n_stroma,
        'n_tumor_epi': n_tumor_epi,
        'n_intra_stroma': n_intra_stroma,
        'n_normal_stroma': n_normal_stroma,
        'pct_cancer': (adata.obs['histology'] == 'cancer_cancer').mean() * 100 if 'histology' in adata.obs.columns else 0,
    })
    print(f"  {sid}: {adata.n_obs} spots")

stats_df = pd.DataFrame(sample_stats)
stats_df['deident'] = stats_df['deident'].astype(str)
meta = meta.merge(stats_df, on='deident', how='left')

# ================================================================
# Build Table 1
# ================================================================

groups = {
    'Total': meta,
    'No Mets': meta[meta['mets_binary'] == 'No_Mets'],
    'Any Mets': meta[meta['mets_binary'] == 'Any_Mets'],
}

def fmt_median_range(series):
    s = series.dropna()
    if len(s) == 0: return "—"
    return f"{s.median():.0f} ({s.min():.0f}-{s.max():.0f})"

def fmt_mean_sd(series):
    s = series.dropna()
    if len(s) == 0: return "—"
    return f"{s.mean():.1f} ({s.std():.1f})"

def fmt_n_pct(df, col, value):
    n = (df[col] == value).sum()
    pct = n / len(df) * 100 if len(df) > 0 else 0
    return f"{n} ({pct:.0f})"

def fmt_sum(series):
    return f"{int(series.sum()):,}"

def fmt_median_range_float(series, decimals=1):
    s = series.dropna()
    if len(s) == 0: return "—"
    return f"{s.median():.{decimals}f} ({s.min():.{decimals}f}-{s.max():.{decimals}f})"

rows = []

# --- DEMOGRAPHICS ---
rows.append(('DEMOGRAPHICS', {g: '' for g in groups}))

rows.append(('Age (years)', {g: fmt_median_range(df['age']) for g, df in groups.items()}))

rows.append(('Female (%)', {g: fmt_n_pct(df, 'sex', 'F') for g, df in groups.items()}))

rows.append(('Tumor Grade (%)', {g: '' for g in groups}))
rows.append(('     Well Differentiated (Grade 1)',
    {g: fmt_n_pct(df, 'Grade', 1) for g, df in groups.items()}))
rows.append(('     Moderately Differentiated (Grade 2)',
    {g: fmt_n_pct(df, 'Grade', 2) for g, df in groups.items()}))
rows.append(('     Poorly Differentiated (Grade 3)',
    {g: fmt_n_pct(df, 'Grade', 3) for g, df in groups.items()}))

rows.append(('Lymph-Node Status (%)', {g: '' for g in groups}))
rows.append(('     N0', {g: fmt_n_pct(df, 'node_stage', 'N0') for g, df in groups.items()}))
rows.append(('     N+', {g: f"{len(df[df['node_stage'] != 'N0'])} ({len(df[df['node_stage'] != 'N0'])/max(len(df),1)*100:.0f})" for g, df in groups.items()}))

rows.append(('Distant Metastasis Status (%)', {g: '' for g in groups}))
rows.append(('     M0', {g: f"{(~df['has_distant']).sum()} ({(~df['has_distant']).sum()/max(len(df),1)*100:.0f})" for g, df in groups.items()}))
rows.append(('     M1', {g: f"{df['has_distant'].sum()} ({df['has_distant'].sum()/max(len(df),1)*100:.0f})" for g, df in groups.items()}))

rows.append(('Site in Colon (%)', {g: '' for g in groups}))
for site in sorted(meta['site'].dropna().unique()):
    rows.append((f'     {site}', {g: fmt_n_pct(df, 'site', site) for g, df in groups.items()}))

rows.append(('MLH1 Status (%)', {g: '' for g in groups}))
rows.append(('     MLH1 Deficient', {g: fmt_n_pct(df, 'MLH1', 0) for g, df in groups.items()}))
rows.append(('     MLH1 Intact', {g: fmt_n_pct(df, 'MLH1', 1) for g, df in groups.items()}))

# --- SPATIAL TRANSCRIPTOMICS ---
rows.append(('', {g: '' for g in groups}))
rows.append(('VISIUM SPATIAL TRANSCRIPTOMICS', {g: '' for g in groups}))

rows.append(('Total spots', {g: fmt_sum(df['n_spots']) for g, df in groups.items()}))

rows.append(('Spots per sample, median (range)',
    {g: fmt_median_range(df['n_spots']) for g, df in groups.items()}))

rows.append(('Genes profiled per spot', {g: f"{int(df['n_genes'].median()):,}" if len(df) > 0 else "—" for g, df in groups.items()}))

rows.append(('Estimated cells per spot, mean (SD)',
    {g: fmt_mean_sd(df['mean_cells_per_spot']) for g, df in groups.items()}))

# --- SPATIAL REGIONS ---
rows.append(('', {g: '' for g in groups}))
rows.append(('SPATIAL REGIONS (500 um boundary)', {g: '' for g in groups}))

rows.append(('Central Tumor spots, total',
    {g: fmt_sum(df['n_ct']) for g, df in groups.items()}))
rows.append(('Central Tumor spots per sample, median (range)',
    {g: fmt_median_range(df['n_ct']) for g, df in groups.items()}))

rows.append(('Invasive Margin spots, total',
    {g: fmt_sum(df['n_im']) for g, df in groups.items()}))
rows.append(('Invasive Margin spots per sample, median (range)',
    {g: fmt_median_range(df['n_im']) for g, df in groups.items()}))

rows.append(('Stroma spots, total',
    {g: fmt_sum(df['n_stroma']) for g, df in groups.items()}))
rows.append(('Stroma spots per sample, median (range)',
    {g: fmt_median_range(df['n_stroma']) for g, df in groups.items()}))

# --- REFINED TISSUE CLASSIFICATION ---
rows.append(('', {g: '' for g in groups}))
rows.append(('REFINED TISSUE CLASSIFICATION (C2L)', {g: '' for g in groups}))

rows.append(('Tumor epithelial spots, total',
    {g: fmt_sum(df['n_tumor_epi']) for g, df in groups.items()}))

rows.append(('Intratumoral stroma spots, total',
    {g: fmt_sum(df['n_intra_stroma']) for g, df in groups.items()}))

rows.append(('% cancer spots classified as intratumoral stroma',
    {g: fmt_median_range_float(df['n_intra_stroma'] / (df['n_tumor_epi'] + df['n_intra_stroma']).replace(0, np.nan) * 100) for g, df in groups.items()}))

rows.append(('Normal stroma spots, total',
    {g: fmt_sum(df['n_normal_stroma']) for g, df in groups.items()}))

# ================================================================
# Print and Save
# ================================================================

n_total = len(meta)
n_no = len(groups['No Mets'])
n_any = len(groups['Any Mets'])

col_headers = {
    'Total': f'Total (N={n_total})',
    'No Mets': f'No Nodal or Distant Metastasis (N={n_no})',
    'Any Mets': f'Nodal or Distant Metastasis (N={n_any})',
}

print(f"\n{'='*110}")
print(f"Table 1: Cohort Characteristics — Visium Spatial Transcriptomics Samples")
print(f"{'='*110}")
print(f"{'':45s} {col_headers['Total']:>20s} {col_headers['No Mets']:>22s} {col_headers['Any Mets']:>22s}")
print("-" * 110)

for var_name, vals in rows:
    print(f"{var_name:45s} {vals.get('Total',''):>20s} {vals.get('No Mets',''):>22s} {vals.get('Any Mets',''):>22s}")

# Save CSV
out_rows = []
for var_name, vals in rows:
    out_rows.append({
        'Variable': var_name,
        col_headers['Total']: vals.get('Total', ''),
        col_headers['No Mets']: vals.get('No Mets', ''),
        col_headers['Any Mets']: vals.get('Any Mets', ''),
    })

table_df = pd.DataFrame(out_rows)
out_path = os.path.join(OUTPUT_DIR, 'Table1_Visium_Cohort.csv')
table_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

Loading spatial data per sample...
  0: 6658 spots
  10: 6138 spots
  100: 7194 spots
  102: 6847 spots
  106: 7088 spots
  107: 7057 spots
  11: 7353 spots
  111: 7046 spots
  116: 8238 spots
  117: 4778 spots
  119: 7191 spots
  122: 5903 spots
  127: 7173 spots
  128: 6661 spots
  13: 5910 spots
  14: 6855 spots
  18: 6319 spots
  28: 6989 spots
  29: 6768 spots
  3: 5571 spots
  30: 6476 spots
  34: 6297 spots
  36: 6873 spots
  38: 6971 spots
  40: 7217 spots
  46: 6826 spots
  49: 6502 spots
  51: 6342 spots
  52: 6718 spots
  6: 6414 spots
  60: 6492 spots
  72: 6706 spots
  73: 6532 spots
  74: 7010 spots
  80: 6526 spots
  81: 7274 spots
  83: 7485 spots
  86: 7480 spots
  91: 6651 spots
  96: 6987 spots
  97: 6142 spots

Table 1: Cohort Characteristics — Visium Spatial Transcriptomics Samples
                                                      Total (N=41) No Nodal or Distant Metastasis (N=19) Nodal or Distant Metastasis (N=22)
----------------------------------------------